# Intel Order book : predicting the next mid-price move 

Trying to take a limit-order book for Intel, cleaning it up and formatting it to be at 1s intervals, and then building a set of features to use to predict whether the mid-price will move up or down using logistic regression.

Data: Nasdap TotalView-ITCH for 30 Dec 2019, rebuilt into a 10-level book (thanks Claude)

Units:  Time is nanoseconds since midnight, US Eastern. 
        Prices are integers in $0.0001, so 596300 is $59.63
        Sizes are shares

## 1. Setup

In [13]:
import pandas as pd
import numpy as np

TICKER = 'INTC'

OPEN = pd.Timedelta('09:30:00').value
CLOSE = pd.Timedelta('16:00:00').value
STEP = pd.Timedelta('1s').value             #spacing of the resampled book

## 2. Load the data

The msg stream is not uniform, events arrive at random times one after another. The book does not have timestamps, row i of the 'book' is the state straight after event i in 'msg'. 

### Data key

**`msg`**: one row per order book event.

| Column     | Meaning                                                                                                    |
| ---------- | ---------------------------------------------------------------------------------------------------------- |
| `time`     | Nanoseconds since midnight, US Eastern                                                                     |
| `event`    | `add`, `cancel` (part of an order), `delete` (whole order), `execute`, `hidden_execute`, `cross` (auction) |
| `order_id` | Exchange order reference                                                                                   |
| `side`     | +1 buy order, -1 sell order, 0 not reported. On an `execute` it is the resting order's side                |
| `price`    | Price of the order affected, in $0.0001                                                                    |
| `size`     | Shares added, removed or traded                                                                            |
| `itch`     | See below                                                                                                  |

**`itch` **: the raw Nasdaq message each row came from, and the `event` it became.

| `itch` | Nasdaq message                                        | `event`                      |
| ------ | ----------------------------------------------------- | ---------------------------- |
| A      | Add order                                             | `add`                        |
| F      | Add order, naming the firm that placed it             | `add`                        |
| X      | Cancel part of an order                               | `cancel`                     |
| D      | Delete a whole order                                  | `delete`                     |
| E      | Order executed at its own price                       | `execute`                    |
| C      | Order executed at a different price (mostly auctions) | `execute`                    |
| U      | Replace an order with a new one                       | `delete` row, then `add` row |
| P      | Trade against a hidden order                          | `hidden_execute`             |
| Q      | Opening or closing auction trade                      | `cross`                      |

**`book`**: the book straight after each event, 10 levels per side.

| Column                       | Meaning                                                  |
| ---------------------------- | -------------------------------------------------------- |
| `ask_price_k`, `bid_price_k` | Price at level k, in \$0.0001. Level 1 is the best price |
| `ask_size_k`, `bid_size_k`   | Total shares resting at that level                       |

In [22]:
msg = pd.read_parquet("data/INTC_2019-12-30_messages.parquet")
book = pd.read_parquet("data/INTC_2019-12-30_book.parquet")
assert len(msg) == len(book)

book['time'] = msg['time']

book.shape

(792985, 41)

In [23]:
msg.head()

,time,event,order_id,side,price,size,itch
0,14742473605633,add,118530,1,602500,200,A
1,14742499707646,delete,118530,1,602500,200,D
2,16503897501343,add,324054,1,600800,3,A
3,22821696579852,delete,324054,1,600800,3,D
4,24161597257912,add,1227342,1,589400,1000,A


## 3. Resample the LOB to a regular grid of time (STEP)

We want to have the book at uniform points in time, the events stream (msg level stream) is not uniform and can have bursts of liquidity. 

In [24]:
grid = pd.DataFrame({'time': np.arange(OPEN, CLOSE, STEP)})
book_grid = pd.merge_asof(grid, book, on = 'time')

assert len(book_grid) == (CLOSE - OPEN) // STEP
assert book_grid.notna().all().all()
assert (book_grid['ask_price_1'] > book_grid['bid_price_1']).all()

book_grid[['time', 'bid_price_1', 'bid_size_1', 'ask_price_1', 'ask_size_1']].head()


,time,bid_price_1,bid_size_1,ask_price_1,ask_size_1
0,34200000000000,599700.0,100,600900.0,200
1,34201000000000,599600.0,4000,600800.0,11
2,34202000000000,600000.0,1127,600400.0,300
3,34203000000000,600000.0,1215,600500.0,801
4,34204000000000,600100.0,200,600500.0,633


## 4. Features

Create a new dataframe for the features that we will use to ultimately make our predictions. We want to have features for every state of the book at 1s granularity.

Features may only use information looking at it's own rows or earlier.

Features are simple and intuitive like mid-price, volatility, spread... as well as some inspired by this blog by Headlands technologies (impulse etc) https://blog.headlandstech.com/2017/08/03/quantitative-trading-summary/

In [25]:
features = pd.DataFrame(index = book_grid.index)